In [4]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

In [9]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [18]:
len(sqlite_index.search('docker'))

3

In [14]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=  sqlite_index,
    llm_client=ollama_client,
)

In [15]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. 

However, please note that if you want to receive a certificate, you need to submit your project while submissions are still being accepted. 

Additionally, you do not need to formally register to participate; you can start learning and submitting homework while the submission form is open.


In [4]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=ollama_client,
    instructions=custom_instructions,
)


In [5]:
assistant.rag("How do I get a certificate?")


'Based on the provided context, to get a certificate, you must meet the following requirements:\n\n*   **Enroll in a Live Cohort:** You cannot get a certificate from self-paced mode. You must finish the course with a "live" cohort.\n*   **Pass the Capstone:** You need to pass the Capstone project to receive the certificate.\n*   **Complete Peer Review:** As part of the live cohort, you must peer-review 3 capstone projects (this is not possible in self-paced mode once the course form is closed).\n*   **Update Your Name (Optional):** By default, you are assigned a random nickname. If you want your official name (from a passport, ID card, or driver\'s license) to appear on the certificate instead, you must update your course profile.'

In [6]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course. However, please note the following regarding certificates based on the provided information:\n\n*   **Certificate Requirement:** If you want to receive a certificate, you must submit your project while submissions are still being accepted.\n*   **Cohort Requirement:** Certificates are only available if you finish the course with a "live" cohort; they are not awarded for the self-paced mode.\n*   **Homework:** Homework is not mandatory to join or get a certificate (passing the Capstone project is required for the certificate).'

## RAG with Elasicsearch

In [1]:
from ingest import load_faq_data
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [2]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 79 documents


In [3]:
from ingest import build_es_index
from rag_helper import ElasticRAG

es_client = build_es_index(documents)

In [6]:
rag = ElasticRAG(
    index=es_client,
    llm_client=ollama_client
)

rag.rag("How do I join the course?")

'Based on the provided context, here is how you can join the course:\n\n*   **Start Immediately:** You can begin learning and submitting homework while the form is open without formally registering.\n*   **Automatic Acceptance:** You are automatically accepted upon registration (or by starting), and you do not need to wait for a confirmation email.\n*   **Certificate Requirement:** If you want to receive a certificate, you must submit your project while submissions are still being accepted.\n*   **Registration Purpose:** Registration is primarily to gauge interest before the start date, though it is not checked against a strict list.'

### for later usage after indexing

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

In [2]:
es.search(
    index="course-faq",
    query={
        "match": {"question": "what is RAG"}
    }
)

ObjectApiResponse({'took': 69, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 200, 'relation': 'eq'}, 'max_score': 6.535276, 'hits': [{'_index': 'course-faq', '_id': '3NSzq54Bj3JQH7MA2pTo', '_score': 6.535276, '_source': {'id': 'c6fc2d4d11', 'course': 'llm-zoomcamp', 'section': 'Module 2: Agents', 'question': 'Connection refused error on prompting the ollam RAG?', 'answer': 'If you encounter this error while doing the homework, you can resolve it by restarting the Ollama server using the following command:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nMake sure to rerun the cell containing `ollama serve` if you stop and restart the notebook cell.'}}, {'_index': 'course-faq', '_id': 'SdSzq54Bj3JQH7MAgZOO', '_score': 6.51669, '_source': {'id': 'c19b485c07', 'course': 'machine-learning-zoomcamp', 'section': 'Module 2. Machine Learning for Regression', 'question': 'What is standard deviation?', 'answer': '<{

In [5]:
from rag_helper import ElasticRAG

rag = ElasticRAG(
    index=es,
    llm_client=ollama_client
)

rag.rag("How do I join the course?")

'Based on the provided context, here is how to join the course:\n\n*   **No Formal Registration Required:** You do not need to wait for a confirmation email. You are accepted, and you can simply start learning and submitting homework immediately while the form is still open.\n*   **Certificate Requirement:** While you can join without registering, if you want to receive a certificate, you must submit your project while the course is still accepting submissions.\n*   **Account Setup:** If you do set up an account, you will be automatically assigned a random name on the leaderboard, but you can edit your display name in the course profile.'